In [75]:
from pyspark.sql import SparkSession

# 1. Inicialização da Sessão Spark
spark = SparkSession.builder \
    .appName("Telecom-PDD-Gold-AgingList") \
    .getOrCreate()

# Definindo caminhos 
silver_path = "s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/02-trusted/"

gold_path = "s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/03-refined/"


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [76]:
# 2. Leitura das tabelas da camada Silver e criação de views temporárias
df_clientes=spark.read.parquet(f"{silver_path}01-clientes/")
df_faturas=spark.read.parquet(f"{silver_path}02-faturas/")
df_pagamentos=spark.read.parquet(f"{silver_path}03-pagamentos/")

df_clientes.createOrReplaceTempView("df_clientes")
df_faturas.createOrReplaceTempView("df_faturas")
df_pagamentos.createOrReplaceTempView("df_pagamentos")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [77]:
# 3. Processamento Gold: Cruzamento (Left Join) e Cálculo da Aging List via Spark SQL
# Definindo uma data de processamento simulada para o fechamento do mês (ex: 31/03/2026)
data_processamento_fechamento = "2026-03-31"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [78]:
query_aging_list = f"""
    WITH faturas_em_aberto AS (
        SELECT 
            f.fatura_id,
            f.cliente_id,
            f.valor_fatura,
            f.data_vencimento,
            c.plano,
            to_date('{data_processamento_fechamento}') as data_proc
        FROM df_faturas f
        JOIN df_clientes c ON f.cliente_id = c.cliente_id
        LEFT JOIN df_pagamentos p ON f.fatura_id = p.fatura_id
        WHERE p.fatura_id IS NULL -- Filtra apenas faturas que não possuem registro de pagamento (em aberto)
          AND f.data_vencimento < to_date('{data_processamento_fechamento}')
    ),
    aging_calculado AS (
        SELECT 
            fatura_id,
            cliente_id,
            plano,
            valor_fatura,
            data_vencimento,
            data_proc,
            DATEDIFF(data_proc, data_vencimento) as dias_atraso,
            -- Aplicando as regras de faixas de atraso (Aging List)
            CASE WHEN DATEDIFF(data_proc, data_vencimento) <= 30 THEN 1 ELSE 0 END as flg_1_30,
            CASE WHEN DATEDIFF(data_proc, data_vencimento) > 30 AND DATEDIFF(data_proc, data_vencimento) <= 60 THEN 1 ELSE 0 END as flg_31_60,
            CASE WHEN DATEDIFF(data_proc, data_vencimento) > 60 AND DATEDIFF(data_proc, data_vencimento) <= 90 THEN 1 ELSE 0 END as flg_61_90,
            CASE WHEN DATEDIFF(data_proc, data_vencimento) > 90 THEN 1 ELSE 0 END as flg_90_mais
        FROM faturas_em_aberto
    )
    SELECT * FROM aging_calculado
"""

# Executando a query e criando o DataFrame Gold
df_aging_gold = spark.sql(query_aging_list)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [79]:
df_aging_gold.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+----------+-----------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
| fatura_id|cliente_id|      plano|valor_fatura|data_vencimento| data_proc|dias_atraso|flg_1_30|flg_31_60|flg_61_90|flg_90_mais|
+----------+----------+-----------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
|FAT_000229| CLI_00482|Fibra 500MB|      129.90|     2024-02-13|2026-03-31|        777|       0|        0|        0|          1|
|FAT_000315| CLI_00245|Fibra 300MB|       99.90|     2024-02-07|2026-03-31|        783|       0|        0|        0|          1|
|FAT_000420| CLI_00594|Fibra 300MB|       99.90|     2024-02-24|2026-03-31|        766|       0|        0|        0|          1|
|FAT_000848| CLI_00125|Fibra 300MB|       99.90|     2024-02-01|2026-03-31|        789|       0|        0|        0|          1|
|FAT_000910| CLI_00842|Fibra 500MB|      129.90|     2024-02-21|2026-03-31|        769|       0| 

In [ ]:
+----------+----------+-------------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
| fatura_id|cliente_id|        plano|valor_fatura|data_vencimento| data_proc|dias_atraso|flg_1_30|flg_31_60|flg_61_90|flg_90_mais|
+----------+----------+-------------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
|FAT_000062| CLI_01242|Pós-Pago 50GB|       89.90|     2024-10-07|2026-03-31|        540|       0|        0|        0|          1|
|FAT_000131| CLI_01388|  Fibra 300MB|       99.90|     2024-10-17|2026-03-31|        530|       0|        0|        0|          1|
|FAT_000481| CLI_00919|Pós-Pago 50GB|       89.90|     2024-10-27|2026-03-31|        520|       0|        0|        0|          1|
|FAT_000899| CLI_00717|Pós-Pago 50GB|       89.90|     2024-10-29|2026-03-31|        518|       0|        0|        0|          1|
|FAT_001041| CLI_00652|  Fibra 500MB|      129.90|     2024-10-04|2026-03-31|        543|       0|        0|        0|          1|
|FAT_001336| CLI_00529|  Fibra 300MB|       99.90|     2024-10-14|2026-03-31|        533|       0|        0|        0|          1|
|FAT_001660| CLI_00910|  Fibra 300MB|       99.90|     2024-10-04|2026-03-31|        543|       0|        0|        0|          1|
|FAT_002031| CLI_00305|Pós-Pago 50GB|       89.90|     2024-10-16|2026-03-31|        531|       0|        0|        0|          1|
|FAT_002061| CLI_00871|  Fibra 300MB|       99.90|     2024-10-23|2026-03-31|        524|       0|        0|        0|          1|
|FAT_002164| CLI_00234|  Fibra 500MB|      129.90|     2024-10-11|2026-03-31|        536|       0|        0|        0|          1|
|FAT_002228| CLI_01052|  Fibra 500MB|      129.90|     2024-10-30|2026-03-31|        517|       0|        0|        0|          1|
|FAT_002289| CLI_00278|  Fibra 500MB|      129.90|     2024-10-14|2026-03-31|        533|       0|        0|        0|          1|
|FAT_002585| CLI_00387|  Fibra 300MB|       99.90|     2024-10-17|2026-03-31|        530|       0|        0|        0|          1|
|FAT_002747| CLI_00724|Pós-Pago 50GB|       89.90|     2024-10-03|2026-03-31|        544|       0|        0|        0|          1|
|FAT_002925| CLI_00883|  Fibra 300MB|       99.90|     2024-10-31|2026-03-31|        516|       0|        0|        0|          1|
|FAT_003004| CLI_01279|  Fibra 300MB|       99.90|     2024-10-26|2026-03-31|        521|       0|        0|        0|          1|
|FAT_003138| CLI_00350|  Fibra 500MB|      129.90|     2024-10-28|2026-03-31|        519|       0|        0|        0|          1|
|FAT_003263| CLI_00146|  Fibra 300MB|       99.90|     2024-10-19|2026-03-31|        528|       0|        0|        0|          1|
|FAT_003455| CLI_01391|  Fibra 300MB|       99.90|     2024-10-23|2026-03-31|        524|       0|        0|        0|          1|
|FAT_003547| CLI_00226|  Fibra 300MB|       99.90|     2024-10-09|2026-03-31|        538|       0|        0|        0|          1|
+----------+----------+-------------+------------+---------------+----------+-----------+--------+---------+---------+-----------+

In [80]:
df_aging_gold.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

1502

In [81]:
# 4. Salvando a tabela consolidada em formato Parquet na Camada Gold
df_aging_gold.write \
    .mode("overwrite") \
    .parquet(f"{gold_path}aging_list_detalhada/")

print("Tabela Gold 'aging_list_detalhada' gerada e salva com sucesso!")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Tabela Gold 'aging_list_detalhada' gerada e salva com sucesso!

In [87]:
# Registrando a view para validação rápida
df_aging_gold.createOrReplaceTempView("tb_gold_aging")

# 5. Visualizando uma amostra do resultado
print("\n Amostra da Aging List Detalhada:")
spark.sql("SELECT fatura_id, cliente_id, valor_fatura, dias_atraso, flg_1_30, flg_31_60, flg_61_90, flg_90_mais FROM tb_gold_aging").show(10)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


 Amostra da Aging List Detalhada:
+----------+----------+------------+-----------+--------+---------+---------+-----------+
| fatura_id|cliente_id|valor_fatura|dias_atraso|flg_1_30|flg_31_60|flg_61_90|flg_90_mais|
+----------+----------+------------+-----------+--------+---------+---------+-----------+
|FAT_000229| CLI_00482|      129.90|        777|       0|        0|        0|          1|
|FAT_000315| CLI_00245|       99.90|        783|       0|        0|        0|          1|
|FAT_000420| CLI_00594|       99.90|        766|       0|        0|        0|          1|
|FAT_000848| CLI_00125|       99.90|        789|       0|        0|        0|          1|
|FAT_000910| CLI_00842|      129.90|        769|       0|        0|        0|          1|
|FAT_000921| CLI_00802|       99.90|        780|       0|        0|        0|          1|
|FAT_001545| CLI_01342|      129.90|        767|       0|        0|        0|          1|
|FAT_001786| CLI_01254|       99.90|        777|       0|        